# TP 1 — Premier CNN sur MNIST

**Objectifs**

- Charger MNIST via `torchvision.datasets`.
- Construire un petit CNN à 2 blocs convolutifs.
- Écrire une boucle d'entraînement complète.
- Évaluer le modèle et visualiser quelques erreurs.

**Durée indicative : 60 minutes.**

> Note : on travaille sur un **sous-ensemble** de MNIST (3 000 train / 1 000 test) pour que tout tourne en quelques minutes sur CPU. Les valeurs d'accuracy seront un peu en-dessous de ce que donne MNIST complet (~99 %).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = "data"
print("device:", DEVICE)

## Exercice 1 — Charger MNIST en sous-ensemble

1. Crée un transform `ToTensor` (qui passe en `float32` dans `[0, 1]` et range en `(C, H, W)`).
2. Charge `datasets.MNIST` train et test (`download=True`).
3. Restreins le train aux 3 000 premiers exemples et le test aux 1 000 premiers (`Subset`).
4. Crée deux `DataLoader` avec `batch_size=64`, shuffle train uniquement.
5. Affiche 8 exemples du train avec leur label.

In [ ]:
# TODO


## Exercice 2 — Construire le CNN

Construis la classe suivante :

```python
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        # conv1 : 1 → 16, kernel 3, padding 1
        # pool 2×2
        # conv2 : 16 → 32, kernel 3, padding 1
        # pool 2×2
        # fc1 : 32*7*7 → 64
        # fc2 : 64 → n_classes
        ...
    def forward(self, x):
        ...
```

Vérifie que `model(torch.zeros(1, 1, 28, 28)).shape == (1, 10)`.

<details>
<summary>💡 Coup de pouce — construire un petit CNN PyTorch</summary>

**🎯 But :** définir un CNN basique qui mappe `(B, 1, 28, 28)` → `(B, 10)` pour MNIST.

**Architecture cible**

```
Conv(1→16, 3×3, padding=1) → ReLU → MaxPool(2)    # 28×28 → 14×14
Conv(16→32, 3×3, padding=1) → ReLU → MaxPool(2)   # 14×14 → 7×7
Flatten → Linear(32*7*7, 64) → ReLU → Linear(64, 10)
```

**Le code**

```python
import torch.nn as nn
import torch.nn.functional as F

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1   = nn.Linear(32 * 7 * 7, 64)
        self.fc2   = nn.Linear(64, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))   # (B, 16, 28, 28)
        x = F.max_pool2d(x, 2)       # (B, 16, 14, 14)
        x = F.relu(self.conv2(x))   # (B, 32, 14, 14)
        x = F.max_pool2d(x, 2)       # (B, 32, 7, 7)
        x = x.flatten(start_dim=1)   # (B, 32*7*7) = (B, 1568)
        x = F.relu(self.fc1(x))      # (B, 64)
        x = self.fc2(x)              # (B, 10) — logits
        return x
```

**Les règles à retenir**

⚠️ **`padding=1`** avec `kernel_size=3` → la convolution **conserve la taille spatiale**. Sans padding, on perd 2 pixels par côté.

⚠️ **`MaxPool2d(2)`** divise H et W par 2. Après 2 pools sur 28×28 → 14×14 → 7×7. C'est pour ça que la première FC a `32*7*7` features.

⚠️ **`x.flatten(start_dim=1)`** garde la dim batch (axe 0) et aplatit le reste. `flatten()` sans arg aplatirait TOUT, mélangeant les exemples du batch.

⚠️ **Pas de softmax à la fin** ! On renvoie des **logits**. La `CrossEntropyLoss` de PyTorch applique le softmax interne pour la numérique.

**Compter les paramètres**

```python
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} paramètres")
```

Pour ce modèle : ~108 000 params. Suffisant pour MNIST, ridiculement peu vs ResNet (25M).

</details>

In [ ]:
# TODO


## Exercice 3 — Boucle d'entraînement

Écris une fonction `train_one_epoch(model, loader, opt, loss_fn)` qui parcourt le loader une fois, fait `forward / loss / backward / step`, et retourne la perte moyenne et l'accuracy.

Écris aussi `evaluate(model, loader, loss_fn)` qui calcule perte et accuracy en mode `eval` sans gradients.

Entraîne pendant 3 époques avec `Adam(lr=1e-3)` et `CrossEntropyLoss`. Affiche les courbes de perte et d'accuracy train / test.

<details>
<summary>💡 Coup de pouce — boucle d'entraînement PyTorch</summary>

**🎯 But :** maîtriser la séquence canonique des 5 étapes du training step.

**Setup**

```python
import torch.optim as optim
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SmallCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
```

**La boucle d'entraînement (5 étapes par batch)**

```python
model.train()
for epoch in range(3):
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        # 1️⃣ zero_grad
        optimizer.zero_grad()
        # 2️⃣ forward
        logits = model(x)
        # 3️⃣ loss
        loss = loss_fn(logits, y)
        # 4️⃣ backward
        loss.backward()
        # 5️⃣ step
        optimizer.step()
```

**Les 5 étapes — ne jamais en oublier une**

| Étape | Effet | Conséquence si oubliée |
|---|---|---|
| `zero_grad()` | Reset les gradients à 0 | **Bug subtil** : gradients accumulés entre batchs → updates aberrants |
| `model(x)` | Forward pass | (impossible d'oublier) |
| `loss_fn(...)` | Calcule le scalaire de perte | (impossible) |
| `loss.backward()` | Backward pass — calcule les ∂loss/∂param | Pas de gradients → step() ne fait rien |
| `optimizer.step()` | Applique les gradients sur les params | Le modèle n'apprend rien |

**Boucle d'évaluation**

```python
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
acc = correct / total
print(f"Test accuracy: {acc:.4f}")
```

**Pourquoi `model.eval()` et `torch.no_grad()`** ?

- `model.eval()` : désactive `Dropout` et fige `BatchNorm` (utilise stats du train, pas du batch courant). Important pour reproductibilité de l'inférence.
- `torch.no_grad()` : désactive autograd → pas de stockage des activations pour le backward. **Économise beaucoup de mémoire** et accélère.

**`.item()`** : convertit un tenseur scalaire 0-D en float Python pur. Sans ça, vous accumuleriez des tenseurs sur GPU → memory leak.

</details>

In [ ]:
# TODO


## Exercice 4 — Visualiser les erreurs

Trouve les indices où la prédiction du modèle diffère du label, et affiche 6 erreurs avec un titre `(vrai → prédit)`. Quelles classes sont les plus confondues ?

<details>
<summary>💡 Coup de pouce — analyser les erreurs du CNN</summary>

**🎯 But :** identifier quels chiffres le modèle confond et visualiser des exemples mal classés.

**Récupérer toutes les prédictions sur le test**

```python
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(dim=1)
        all_preds.append(pred.cpu())
        all_labels.append(y.cpu())
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)
```

⚠️ **`.cpu()` avant d'accumuler** : sinon les tenseurs s'empilent en mémoire GPU.

⚠️ **`torch.cat`** concatène les morceaux en un seul tenseur final.

**Indices des erreurs**

```python
wrong = (all_preds != all_labels).nonzero(as_tuple=True)[0]
print(f"{len(wrong)} erreurs sur {len(all_labels)}")
```

`(all_preds != all_labels)` → tenseur booléen. `.nonzero(as_tuple=True)[0]` → indices des True.

**Afficher 6 erreurs typiques**

```python
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for ax, idx in zip(axes, wrong[:6].tolist()):
    img = X_test[idx].squeeze()      # squeeze enlève les dims de taille 1 (channel)
    ax.imshow(img, cmap='gray')
    ax.set_title(f"vrai:{all_labels[idx].item()} | pred:{all_preds[idx].item()}", fontsize=9)
    ax.axis('off')
```

`.squeeze()` enlève les dimensions de taille 1. Une image MNIST tensorielle est `(1, 28, 28)` → après squeeze `(28, 28)` → affichable par `imshow`.

**Confusions typiques sur MNIST**

- **4 ↔ 9** : très similaires sans la barre du 4 fermée
- **3 ↔ 8** : les boucles ressemblent
- **7 ↔ 1** : un 7 mal écrit ressemble à un 1
- **5 ↔ 6 ↔ 0** : si écriture pas nette

C'est le test ultime : votre CNN se trompe-t-il sur des cas **graphiquement ambigus** (acceptable) ou sur des cas **clairs** (problème de capacité du modèle) ?

</details>

In [ ]:
# TODO
